In [1]:
import sys
from pyprojroot import here
sys.path.insert(0, str(here()))

In [2]:
import copy
import optuna
import numpy as np
from transformers import AutoTokenizer

from src.train import train_single_fold, tokenize_df
from src.config import load_config
from src.data_holdout import get_pooled_df, get_fold_from_disk

import yaml
import torch
from datasets import Dataset
from pyprojroot import here
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

from sklearn.metrics import f1_score

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
MODEL_NAME = "roberta-base"

In [4]:
#open shared settings
with open(here("configs/base.yaml"), "r") as f:
    base_data = yaml.full_load(f)

#get df
full_df = get_pooled_df()

# Same pattern as collaborator: all folds
train_folds, val_folds = get_fold_from_disk(full_df)

#load tokenizer and write tokenize function
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fold(df):
    ds = Dataset.from_pandas(df, preserve_index=False)

    def _tokenize(examples):
        return tokenizer(
            examples["abstract_conclusion"],
            examples["press_release_conclusion"],
            truncation=True,
            padding="max_length",
            max_length=base_data["max_length"],
        )

    cols_to_remove = [c for c in ds.column_names if c != "exaggeration_label"]
    ds = ds.map(_tokenize, batched=True, remove_columns=cols_to_remove)
    ds = ds.rename_column("exaggeration_label", "labels")
    ds.set_format("torch")
    return ds

#tokenize all folds
train_ds = [tokenize_fold(df) for df in train_folds]
val_ds = [tokenize_fold(df) for df in val_folds]

for i in range(len(train_ds)):
    print(f"Fold {i}: train={len(train_ds[i])} val={len(val_ds[i])}")

# define metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"macro_f1": f1_score(labels, preds, average="macro")}

#Freeze encoder, train classifier head only
def freeze_base(model):
    for name, param in model.named_parameters():
        if "classifier" not in name:
            param.requires_grad = False

#Set up Optuna objective
def objective(trial):
    # Match collaborator's structure, but use a frozen-head-appropriate LR range
    lr = trial.suggest_float("learning_rate", 1e-3, 1e-1, log=True) #higher LR for classification head only
    batch_size = trial.suggest_categorical("per_device_train_batch_size", [8])
    weight_decay = trial.suggest_float("weight_decay", 0.0, 0.1)
    warmup_ratio = trial.suggest_float("warmup_ratio", 0.05, 0.2) #empirically found that higher is better for classification head only
    num_epochs = trial.suggest_int("num_train_epochs", 12, 30) #not as many epochs needed for classification head only

    f1_scores = []

    for fold in range(5):
        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME,
            num_labels=3
        )
        freeze_base(model)

        training_args = TrainingArguments(
            output_dir=str(here(f"results/optuna/roberta_frozen/trial-{trial.number}/fold-{fold}")),
            eval_strategy=base_data["eval_strategy"],
            save_strategy=base_data["save_strategy"],
            learning_rate=lr,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            weight_decay=weight_decay,
            warmup_ratio=warmup_ratio,
            num_train_epochs=num_epochs,
            load_best_model_at_end=True,
            metric_for_best_model="macro_f1",
            greater_is_better=base_data["greater_is_better"],
            fp16=base_data["fp16"],
            seed=base_data["seed"],
            logging_steps=10,
            report_to=base_data["report_to"],
            save_total_limit=1,
            skip_memory_metrics=True,
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_ds[fold],
            eval_dataset=val_ds[fold],
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
        )

        trainer.train()
        f1_scores.append(trainer.evaluate()["eval_macro_f1"])

        del model
        torch.cuda.empty_cache()

    return float(np.mean(f1_scores))

# Study definition
db_path = here("results/optuna/optuna_study_roberta_frozen.db")
storage_url = f"sqlite:///{db_path}"
n_startup_trials = 5

print(storage_url)

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(
        n_startup_trials=n_startup_trials,
        seed=42,
    ),
    study_name="roberta-frozen-hyperparameter-search",
    storage=storage_url,
    load_if_exists = True
)

study.optimize(objective, n_trials=50)

# Best trials
print(f"\n{'='*60}")
print(f"  Best macro_f1: {study.best_trial.value:.4f}")
print(f"  Best params:   {study.best_trial.params}")
print(f"{'='*60}")


/workspace/exaggeration-detection/src/data_holdout.py:65: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(ENCODED_LABELS)
Map: 100%|██████████| 106/106 [00:00<00:00, 7701.30 examples/s]


Fold 0: train=424 val=106
Fold 1: train=424 val=106
Fold 2: train=424 val=106
Fold 3: train=424 val=106
Fold 4: train=424 val=106
sqlite:////workspace/exaggeration-detection/results/optuna/optuna_study_roberta_frozen.db


[I 2026-03-24 15:40:15,497] Using an existing study with name 'roberta-frozen-hyperparameter-search' instead of creating a new one.
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1213.08it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]             
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architectu

Epoch,Training Loss,Validation Loss,Macro F1
1,1.557702,2.981399,0.118863
2,2.899228,12.625348,0.096774
3,13.864012,9.332547,0.253411
4,16.644543,15.146008,0.118863
5,21.475107,8.668549,0.118863
6,8.764472,3.969499,0.253411


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.66it/s]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.la

...

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1489.13it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consid

Epoch,Training Loss,Validation Loss,Macro F1
1,1.052375,1.137437,0.253411
2,1.628506,1.489048,0.118863
3,2.242084,1.130919,0.311971
4,3.121904,1.175431,0.208859
5,2.055344,2.143419,0.204681
6,3.557074,3.671481,0.118863


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.66it/s]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.la

[I 2026-03-24 18:12:11,773] Trial 101 finished with value: 0.2956945938197305 and parameters: {'learning_rate': 0.03007134459873541, 'per_device_train_batch_size': 8, 'weight_decay': 0.09603702529852587, 'warmup_ratio': 0.16849820247833758, 'num_train_epochs': 23}. Best is trial 34 with value: 0.41547501867315795.



  Best macro_f1: 0.4155
  Best params:   {'learning_rate': 0.04943018624063399, 'per_device_train_batch_size': 8, 'weight_decay': 0.06881360999565503, 'warmup_ratio': 0.19571930271634483, 'num_train_epochs': 24}


In [5]:
print(study.best_trial)
print(study.best_value)
print(study.best_params)

FrozenTrial(number=34, state=<TrialState.COMPLETE: 1>, values=[0.41547501867315795], datetime_start=datetime.datetime(2026, 3, 22, 16, 41, 32, 174557), datetime_complete=datetime.datetime(2026, 3, 22, 16, 44, 2, 490067), params={'learning_rate': 0.04943018624063399, 'per_device_train_batch_size': 8, 'weight_decay': 0.06881360999565503, 'warmup_ratio': 0.19571930271634483, 'num_train_epochs': 24}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'learning_rate': FloatDistribution(high=0.1, log=True, low=0.001, step=None), 'per_device_train_batch_size': CategoricalDistribution(choices=(8,)), 'weight_decay': FloatDistribution(high=0.1, log=False, low=0.0, step=None), 'warmup_ratio': FloatDistribution(high=0.2, log=False, low=0.05, step=None), 'num_train_epochs': IntDistribution(high=30, log=False, low=12, step=1)}, trial_id=35, value=None)
0.41547501867315795
{'learning_rate': 0.04943018624063399, 'per_device_train_batch_size': 8, 'weight_decay': 0.06881360999565503,